# Notebook 03 - Enriquecedor de Tracks

Enriquece tracks com BPM, Camelot key e energia via Spotify e tunebat.

> O enriquecedor e sincrono — sem await necessario.

In [ ]:
import sys
sys.path.insert(0, '../')

import os, logging
from dotenv import load_dotenv
from supabase import create_client
from src.enricher.enricher import Enricher

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
load_dotenv('../.env')

sb       = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_KEY'])
enricher = Enricher(sb, os.environ['SPOTIFY_CLIENT_ID'], os.environ['SPOTIFY_CLIENT_SECRET'])
print('Enricher pronto')

## TESTE: Enriquecer tracks individuais

Valida que o Spotify API esta funcionando.

In [ ]:
testes = [
    ('Astrix', 'Basic'),
    ('Vini Vici', 'The Tribe'),
    ('Artista Inexistente', 'Track Que Nao Existe'),
]

for artist, title in testes:
    print(f'\n--- {artist} - {title} ---')
    r = enricher.enrich_track(artist, title)
    print(f'  Fonte:        {r["source"]}')
    print(f'  Confianca:    {r["confidence"]}')
    print(f'  BPM:          {r["bpm"]}')
    print(f'  Camelot Key:  {r["camelot_key"]}')
    print(f'  Energy:       {r["energy"]}')
    print(f'  Danceability: {r["danceability"]}')

## Enriquecer em lote as tracks sem dados

Rode apos uma coleta grande no notebook 01.

In [ ]:
sem_dados = (
    sb.table('tracks').select('id', count='exact')
    .eq('source', 'unknown').execute().count or 0
)
print(f'Tracks sem dados acusticos: {sem_dados}')

if sem_dados > 0:
    BATCH = 100
    print(f'Enriquecendo ate {min(BATCH, sem_dados)} tracks... (demora alguns minutos)')
    enricher.enrich_all_unenriched(batch_size=BATCH)
    print('Lote concluido!')
else:
    print('Todas as tracks ja tem dados. Nada a fazer.')

## Cobertura de dados acusticos na base

In [ ]:
import pandas as pd

tracks = sb.table('tracks').select('source, confidence').execute().data
if not tracks:
    print('Nenhuma track na base. Rode o notebook 01 primeiro.')
else:
    df = pd.DataFrame(tracks)
    print('=== Por fonte ===')
    print(df['source'].value_counts().to_string())
    print()
    print('=== Por confianca ===')
    print(df['confidence'].value_counts().to_string())
    total     = len(df)
    com_dados = len(df[df['source'] != 'unknown'])
    print(f'\nCobertura: {com_dados}/{total} ({100*com_dados/max(total,1):.1f}%)')

## Buscar dados de uma track na base

In [ ]:
# -- EDITE AQUI --
ARTISTA = 'Astrix'
TITULO  = ''  # Deixe vazio para ver todas as tracks do artista
# ----------------

q = sb.table('tracks').select('*').ilike('artist', f'%{ARTISTA}%')
if TITULO:
    q = q.ilike('title', f'%{TITULO}%')

results = q.limit(20).execute().data
if results:
    for r in results:
        print(f"{r['artist']} - {r['title']}")
        print(f"  BPM: {r['bpm']} | Key: {r['camelot_key']} | Energy: {r['energy']}")
        print(f"  Fonte: {r['source']} | Confianca: {r['confidence']}")
else:
    print('Nenhuma track encontrada.')